# CN-GongWen Benchmark · 一键**完美复现** Colab 脚本

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariskang/SH-PA/blob/main/notebooks/CN_GongWen_Reproduce_Colab.ipynb)

本脚本端到端复现三套**中国党政机关公文**评测数据集（含约一半医疗卫生政策、三级具体分类，以及按 token 分桶的写作测试 prompt）。

**为什么能“完美复现”**：数据生成完全基于 `SHA-256` 哈希、**无随机数、核心无第三方依赖**，
因此在任何环境、任何时间从源码重新生成，结果都与仓库已提交工件**逐字节一致**——本脚本第 6 步会用
`git diff` 当场证明这一点。

流程：配置 → 环境信息 → 克隆 → 安装(含中文字体) → 从零生成 → **确定性比对** → 严格校验 →
单元测试 → 公文分布可视化 → 医疗三级分类可视化 → 证据复核 → 打分器演示 → 分类样例 → 写作测试 prompt(token 分桶) → 审核/纠错。

## 0️⃣ 配置


In [ ]:
REPO_URL = 'https://github.com/pariskang/SH-PA.git'
BRANCH   = 'main'                  # 代码已合并至 main；如需特定分支自行修改
PROFILE  = 'standard'              # mini(快) / standard(提交基准) / full(生产规模)
print('repo =', REPO_URL, '| branch =', BRANCH, '| profile =', PROFILE)

## 1️⃣ 环境信息（可复现性记录）


In [ ]:
import sys, platform, datetime
print('Python :', sys.version.split()[0])
print('Platform:', platform.platform())
print('Time   :', datetime.datetime.now().isoformat(timespec='seconds'))
print('注：核心生成仅依赖标准库，结果与上述环境无关（确定性）。')


## 2️⃣ 克隆仓库（指定分支，可重复运行）


In [ ]:
import os
BASE = '/content' if os.path.isdir('/content') else os.getcwd()
os.chdir(BASE)
if not os.path.isdir('SH-PA'):
    !git clone --branch {BRANCH} {REPO_URL} SH-PA
os.chdir(os.path.join(BASE, 'SH-PA'))
!git checkout -q {BRANCH} 2>/dev/null; git pull -q origin {BRANCH} 2>/dev/null
print('CWD:', os.getcwd())
!git log --oneline -3


## 3️⃣ 安装依赖（含中文字体）

核心生成/校验仅需标准库；此处装 `pytest`、`PyYAML` 以便测试，并装 Noto CJK 字体以正确显示图表中文。


In [ ]:
!pip -q install pytest PyYAML
!apt-get -qq install -y fonts-noto-cjk > /dev/null 2>&1 || true
# 可选：启用 LiteLLM 改写 / Parquet 导出
# !pip -q install 'litellm>=1.0.0' pandas pyarrow
print('依赖安装完成')


## 4️⃣ 配置中文字体（修复图表中文乱码）


In [ ]:
import glob, matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager as fm
cands = []
for pat in ('NotoSansCJK*', 'NotoSerifCJK*', 'wqy*', '*CJK*'):
    cands += glob.glob(f'/usr/share/fonts/**/{pat}', recursive=True)
cjk = None
for p in cands:
    try:
        fm.fontManager.addfont(p); cjk = fm.FontProperties(fname=p).get_name(); break
    except Exception:
        continue
if cjk:
    plt.rcParams['font.sans-serif'] = [cjk, 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('中文字体：', cjk or '未找到（图表中文可能显示为方块，可重跑第3、4步）')


## 5️⃣ （可选）配置 MiniMax API

LiteLLM 仅在**事实护栏**下改写问题/润色播报，不改变文种、字号、日期、密级等关键事实；
因此有无 LLM 复现结果一致。不配置可直接跳过。


In [ ]:
import os
# os.environ['MINIMAX_API_KEY']  = 'sk-...'
# os.environ['MINIMAX_API_BASE'] = 'https://your-openai-compatible-relay/v1'
# os.environ['MINIMAX_MODEL']    = 'MiniMax-M1'
USE_LLM = bool(os.getenv('MINIMAX_API_KEY'))
print('使用 LiteLLM 改写：', USE_LLM)


## 6️⃣ 从零重新生成，并**当场证明逐字节一致**


In [ ]:
import subprocess
cmd = f'python gongwen_benchmark/scripts/generate_benchmarks.py --profile {PROFILE}'
if USE_LLM: cmd += ' --use-litellm'
print('$', cmd); 
!{cmd}
print()
if PROFILE == 'standard' and not USE_LLM:
    out = subprocess.run(['git','status','--porcelain','gongwen_benchmark'], capture_output=True, text=True).stdout.strip()
    if out:
        print('⚠ 与已提交 standard 版存在差异：'); print(out[:1000])
    else:
        print('✅ 完美复现：重新生成的工件与已提交 standard 版本逐字节一致。')
else:
    print(f'profile={PROFILE} 或启用了 LLM，跳过与提交基准(standard)的逐字节比对。')


## 7️⃣ 严格跨文件校验


In [ ]:
import json
rep = subprocess.run(['python','gongwen_benchmark/scripts/validate_artifacts.py'], capture_output=True, text=True)
if rep.returncode != 0 or not rep.stdout.strip():
    print('❌ 校验未通过（断言失败或无 JSON 输出）：')
    print(rep.stderr.strip() or rep.stdout.strip() or '(无输出)')
    raise SystemExit(1)
print(rep.stderr.strip() or '(无断言错误)')
report = json.loads(rep.stdout)
for k in ['q','dataqa','records','q_hard_share','q_trap_diversity','briefing_ungrounded_count',
          'corpus_medical_share','q_medical_share','medical_area_coverage','medical_topic_coverage']:
    print(f'  {k:24s}: {report[k]}')

## 8️⃣ 单元测试


In [ ]:
!pytest -q


## 9️⃣ 公文维度分布可视化


In [ ]:
import json, collections
ROOT = 'gongwen_benchmark'
def load(p): return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]
hidden = load(f'{ROOT}/dataset_1_question_only/questions_with_hidden_metadata.jsonl')
ques   = load(f'{ROOT}/dataset_2_data_qa/questions.jsonl')
qt = collections.Counter(x['question_type'] for x in hidden)
tt = collections.Counter(x['task_type'] for x in ques)
df = collections.Counter(x['difficulty'] for x in hidden)
fig, ax = plt.subplots(1, 3, figsize=(19, 5))
ax[0].barh(list(qt.keys()), list(qt.values())); ax[0].set_title('CN-GongWen-Q 问题类型(11)')
ax[1].barh(list(tt.keys()), list(tt.values())); ax[1].set_title('CN-GongWen-DataQA 任务类型(16)')
ax[2].bar(list(df.keys()), list(df.values())); ax[2].set_title('Q 难度分布')
plt.tight_layout(); plt.show()
print('hard 占比：', round(df['hard']/sum(df.values()), 3))


## 🔟 医疗政策**三级分类**可视化

政策领域（约半医疗）→ 16 个医疗子领域 → 约 105 个具体政策主题。


In [ ]:
import csv, collections
recs = list(csv.DictReader(open(f'{ROOT}/dataset_2_data_qa/records.csv', encoding='utf-8')))
dom  = collections.Counter(r['policy_domain'] for r in recs)
qdom = collections.Counter(x['policy_domain'] for x in hidden)
area = collections.Counter(r['medical_area'] for r in recs if r['medical_area'])
topic= collections.Counter(r['medical_topic'] for r in recs if r['medical_topic'])
fig, ax = plt.subplots(1, 3, figsize=(20, 7))
ax[0].bar(['语料·'+k for k in dom]+['Q·'+k for k in qdom], list(dom.values())+list(qdom.values()))
ax[0].set_title('政策领域分布（各约50%医疗）'); ax[0].tick_params(axis='x', rotation=30)
ax[1].barh(list(area.keys()), list(area.values())); ax[1].set_title('16个医疗子领域·语料发文量')
top = topic.most_common(20)[::-1]
ax[2].barh([t for t,_ in top], [c for _,c in top]); ax[2].set_title('具体政策主题 Top20（共%d个）' % len(topic))
plt.tight_layout(); plt.show()
print('语料医疗占比：', round(dom['医疗卫生']/sum(dom.values()),3),
      '| Q医疗占比：', round(qdom['医疗卫生']/sum(qdom.values()),3),
      '| 子领域：', len(area), '| 具体主题：', len(topic))


## 1️⃣1️⃣ 证据行复核

每道 DataQA 题的证据行必须都存在于 records.csv。


In [ ]:
doc_ids = {r['doc_id'] for r in recs}
ans = load(f'{ROOT}/dataset_2_data_qa/answers.jsonl')
bad = [a['question_id'] for a in ans if not set(a['evidence_rows']) <= doc_ids]
print('记录数：', len(doc_ids), '| 答案数：', len(ans), '| 证据越界题：', len(bad))
assert not bad, bad[:5]
print('✓ 证据行全部可追溯')


## 1️⃣2️⃣ 打分器演示（金标准自评 = 满分）


In [ ]:
from pathlib import Path
from gongwen_benchmark.evaluation.scorer import dataset1_score, dataset2_score
R = Path(ROOT)
s1 = dataset1_score(R/'dataset_1_question_only/questions_with_hidden_metadata.jsonl',
                    R/'dataset_1_question_only/questions_with_hidden_metadata.jsonl')
s2 = dataset2_score(R/'dataset_2_data_qa/answers.jsonl', R/'dataset_2_data_qa/answers.jsonl')
print('Q 自评：', json.dumps(s1, ensure_ascii=False, indent=2))
print('DataQA 自评：', json.dumps(s2, ensure_ascii=False, indent=2))


## 1️⃣3️⃣ 三级分类样例（政策领域 → 医疗子领域 → 具体主题）


In [ ]:
qmap = {json.loads(l)['question_id']: json.loads(l) for l in open(f'{ROOT}/dataset_2_data_qa/questions.jsonl', encoding='utf-8')}
n = 0
for a in ans:
    if qmap[a['question_id']]['task_type'] == 'policy_domain_classification':
        print('Q:', qmap[a['question_id']]['question'][:48])
        print('  →', json.dumps(a['answer_value'], ensure_ascii=False)); n += 1
        if n >= 5: break


## 1️⃣4️⃣ CN-GongWen-Writing：按 token 分桶的公文**写作测试 prompt**（dataset_3）

短/中/长三档**按目标产出 token** 分桶，对标四类规范（《条例》+ GB/T 9704 格式 + GB/T 15834 标点 + GB/T 15835 数字）。测试 prompt 蕴含复杂行文框架与行文规则（标题三要素、层次序数、上行文签发人、请示单一主送、报告不夹带请示、函为平行文）、**内容可执行性**（责任—时限—反馈）与**语言安全**（规避夸大/网络化表述）。有 `MINIMAX_API_KEY` 时由 LLM **一次 10 条**撰写、否则确定性模板（提交即冻结）。rubric / 参考公文为**确定性事实接地**，故 **11 维**金标准自评满分、逐字节可复现。

In [ ]:
# CN-GongWen-Writing（dataset_3）：按目标产出 token 分桶的公文写作测试 prompt
import collections
from pathlib import Path
from gongwen_benchmark.evaluation.scorer import dataset3_writing_score
from gongwen_benchmark.scripts.tokens import estimate_tokens
from gongwen_benchmark.scripts.generate_writing_prompts import LENGTH_BUCKETS as LB
W = f'{ROOT}/dataset_3_writing'
wrub = load(f'{W}/writing_prompts_with_rubric.jsonl')
order = ['short', 'medium', 'long']
bucket = collections.Counter(h['length_bucket'] for h in wrub)
dtc = collections.Counter(h['spec']['doc_type'] for h in wrub)
toks = {b: [estimate_tokens(h['reference_answer']) for h in wrub if h['length_bucket'] == b] for b in order}
fig, ax = plt.subplots(1, 3, figsize=(20, 5))
ax[0].bar([f"{b}\n{LB[b]['target_tokens']}" for b in order], [bucket[b] for b in order])
ax[0].set_title('长度分桶题量（短/中/长·按目标产出 token）')
ax[1].barh(list(dtc.keys()), list(dtc.values())); ax[1].set_title('15 法定文种覆盖')
ax[2].boxplot([toks[b] for b in order]); ax[2].set_xticks([1, 2, 3]); ax[2].set_xticklabels(order)
ax[2].set_title('参考公文实际 token 分布（按桶，均落入目标区间）')
plt.tight_layout(); plt.show()
P = Path(W) / 'writing_prompts_with_rubric.jsonl'
sc = dataset3_writing_score(P, P)
print('题量：', len(wrub), '| 桶：', dict(bucket), '| 文种：', len(dtc),
      '| 医疗占比：', round(sum(h['spec']['policy_domain'] == '医疗卫生' for h in wrub) / len(wrub), 3))
print('写作打分自评（金标准=参考公文，应全 1.0）：', json.dumps(sc, ensure_ascii=False))
print('\n样例测试 prompt（' + wrub[0]['length_bucket'] + '）：\n' + wrub[0]['prompt'][:320])

## 1️⃣5️⃣ CN-GongWen-Audit：公文**审核/纠错**（dataset_4）

给一份**确定性注入了雷区**的公文（含约 1/4 完全合规的对照样本），要求模型找出全部违规；覆盖 11 类违规，按违规类型 precision/recall/F1 + 逐项精确匹配 + 合规件零误报打分。金标准由**独立检测器**校验诚实、逐字节可复现。

In [ ]:
# CN-GongWen-Audit（dataset_4）：公文审核/纠错
import collections
from pathlib import Path
from gongwen_benchmark.evaluation.scorer import dataset4_audit_score
A=f'{ROOT}/dataset_4_audit'
ag=load(f'{A}/audit_tasks_with_gold.jsonl')
vc=collections.Counter(v for h in ag for v in h['violations'])
clean=sum(h['is_clean'] for h in ag)
fig,ax=plt.subplots(1,2,figsize=(18,5))
ax[0].bar(['合规对照','含缺陷'],[clean,len(ag)-clean]); ax[0].set_title('审核样本：合规 vs 含缺陷')
items=vc.most_common()[::-1]
ax[1].barh([k for k,_ in items],[c for _,c in items]); ax[1].set_title('11 类违规分布')
plt.tight_layout(); plt.show()
P=Path(A)/'audit_tasks_with_gold.jsonl'; s=dataset4_audit_score(P,P)
print('题量：',len(ag),'| 合规：',clean,'| 含缺陷：',len(ag)-clean,'| 违规类型覆盖：',len(vc))
print('审核打分自评（金标准=注入集合，应 F1=1.0）：', json.dumps(s, ensure_ascii=False))
flawed=next(h for h in ag if not h['is_clean'])
print('\n样例含缺陷公文（注入违规：%s）：' % '、'.join(flawed['violations']))
print(flawed['flawed_document'][:400])

## ✅ 完美复现完成

你已从源码端到端重建并校验四套公文评测数据集（含按 token 分桶的写作测试 prompt 与审核/纠错任务），并验证了与提交基准的**逐字节一致性**。

用自己的模型评分：把预测写成与金标准同 `question_id` 的 JSONL，运行：

```bash
python gongwen_benchmark/evaluation/scorer.py --dataset q \
  --gold gongwen_benchmark/dataset_1_question_only/questions_with_hidden_metadata.jsonl \
  --pred your_q_predictions.jsonl
python gongwen_benchmark/evaluation/scorer.py --dataset dataqa \
  --gold gongwen_benchmark/dataset_2_data_qa/answers.jsonl \
  --pred your_dataqa_predictions.jsonl
# 写作测试：预测写成 {"question_id": "...", "answer": "<公文全文>"}
python gongwen_benchmark/evaluation/scorer.py --dataset writing \
  --gold gongwen_benchmark/dataset_3_writing/writing_prompts_with_rubric.jsonl \
  --pred your_writing_predictions.jsonl
# 审核/纠错：预测写成 {"question_id": "...", "violations": ["code", ...]}
python gongwen_benchmark/evaluation/scorer.py --dataset audit \
  --gold gongwen_benchmark/dataset_4_audit/audit_tasks_with_gold.jsonl \
  --pred your_audit_predictions.jsonl
```

> 用 LLM 真正撰写这批写作 prompt 并冻结入库：配置 `MINIMAX_API_KEY` 后运行
> `python gongwen_benchmark/scripts/generate_writing_prompts.py --count 90 --use-litellm`（rubric/参考公文仍为确定性，复现一致）。